# InsightGuard AI

## Phase 3: Dataset Setup and Data Understanding

### Objective

This notebook performs a structured data understanding and baseline quality assessment
of the Olist Brazilian E-Commerce dataset.

The analysis includes:

- Dataset inventory
- Schema inspection
- Data type analysis
- Missing value analysis
- Duplicate analysis
- Primary and composite key validation
- Referential integrity validation
- Table relationship analysis
- Initial business KPI analysis

The results of this notebook will serve as the baseline for the following modules:

1. ETL Pipeline
2. Data Quality Engine
3. Opportunity Detection Engine
4. Data Corruption Simulation
5. Scenario Generation
6. Decision Robustness Analysis

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
def find_project_root(start_path: Path) -> Path:
    """
    Locate the InsightGuard-AI project root by searching
    upward for the data/raw directory.
    """
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "data" / "raw").exists():
            return path

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Make sure the dataset is stored in InsightGuard-AI/data/raw/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = PROJECT_ROOT / "data" / "raw"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Data Path    : {DATA_PATH}")
print(f"Notebook Path: {NOTEBOOK_PATH}")

Project Root : C:\Users\Impana\OneDrive\Data Analyst Tutorial\Project\InsightGuard AI
Data Path    : C:\Users\Impana\OneDrive\Data Analyst Tutorial\Project\InsightGuard AI\data\raw
Notebook Path: C:\Users\Impana\OneDrive\Data Analyst Tutorial\Project\InsightGuard AI\notebooks


In [3]:
DATASET_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

print("Expected datasets:")

for dataset_name, filename in DATASET_FILES.items():
    print(f"• {dataset_name:<22} -> {filename}")

Expected datasets:
• customers              -> olist_customers_dataset.csv
• geolocation            -> olist_geolocation_dataset.csv
• order_items            -> olist_order_items_dataset.csv
• order_payments         -> olist_order_payments_dataset.csv
• order_reviews          -> olist_order_reviews_dataset.csv
• orders                 -> olist_orders_dataset.csv
• products               -> olist_products_dataset.csv
• sellers                -> olist_sellers_dataset.csv
• category_translation   -> product_category_name_translation.csv


In [4]:
missing_files = []

for dataset_name, filename in DATASET_FILES.items():
    file_path = DATA_PATH / filename

    if not file_path.exists():
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(f"- {file}" for file in missing_files)
    )

print("All 9 required dataset files are available.")

All 9 required dataset files are available.


In [5]:
def load_datasets(
    data_path: Path,
    dataset_files: Dict[str, str]
) -> Dict[str, pd.DataFrame]:
    """
    Load all Olist CSV files into a dictionary of pandas DataFrames.
    """
    datasets = {}

    for dataset_name, filename in dataset_files.items():
        file_path = data_path / filename

        print(f"Loading {filename}...")

        datasets[dataset_name] = pd.read_csv(file_path)

    return datasets


datasets = load_datasets(DATA_PATH, DATASET_FILES)

print("\nAll datasets loaded successfully.")

Loading olist_customers_dataset.csv...
Loading olist_geolocation_dataset.csv...
Loading olist_order_items_dataset.csv...
Loading olist_order_payments_dataset.csv...
Loading olist_order_reviews_dataset.csv...
Loading olist_orders_dataset.csv...
Loading olist_products_dataset.csv...
Loading olist_sellers_dataset.csv...
Loading product_category_name_translation.csv...

All datasets loaded successfully.


In [6]:
customers = datasets["customers"]
geolocation = datasets["geolocation"]
order_items = datasets["order_items"]
order_payments = datasets["order_payments"]
order_reviews = datasets["order_reviews"]
orders = datasets["orders"]
products = datasets["products"]
sellers = datasets["sellers"]
category_translation = datasets["category_translation"]

print("DataFrames assigned successfully.")

DataFrames assigned successfully.


In [7]:
def create_dataset_inventory(
    datasets: Dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """
    Create a summary of all datasets including
    rows, columns, memory usage, and total cells.
    """
    inventory = []

    for name, df in datasets.items():
        memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

        inventory.append({
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "total_cells": df.shape[0] * df.shape[1],
            "memory_mb": round(memory_mb, 2)
        })

    return (
        pd.DataFrame(inventory)
        .sort_values("rows", ascending=False)
        .reset_index(drop=True)
    )


dataset_inventory = create_dataset_inventory(datasets)

display(dataset_inventory)

,dataset,rows,columns,total_cells,memory_mb
0,geolocation,1000163,5,5000815,129.38
1,order_items,112650,7,788550,35.99
2,order_payments,103886,5,519430,16.23
3,customers,99441,5,497205,26.59
4,orders,99441,8,795528,52.94
5,order_reviews,99224,7,694568,39.12
6,products,32951,9,296559,6.30
7,sellers,3095,4,12380,0.59
8,category_translation,71,2,142,0.01


In [8]:
total_rows = dataset_inventory["rows"].sum()
total_columns = dataset_inventory["columns"].sum()
total_memory = dataset_inventory["memory_mb"].sum()

print("DATASET OVERVIEW")
print("-" * 50)
print(f"Total Datasets : {len(datasets)}")
print(f"Total Rows     : {total_rows:,}")
print(f"Total Columns  : {total_columns:,}")
print(f"Total Memory   : {total_memory:,.2f} MB")

DATASET OVERVIEW
--------------------------------------------------
Total Datasets : 9
Total Rows     : 1,550,922
Total Columns  : 52
Total Memory   : 307.15 MB


In [9]:
schema_records = []

for dataset_name, df in datasets.items():

    for column in df.columns:

        schema_records.append({
            "dataset": dataset_name,
            "column": column,
            "data_type": str(df[column].dtype),
            "non_null_count": int(df[column].notna().sum()),
            "null_count": int(df[column].isna().sum()),
            "unique_values": int(df[column].nunique(dropna=True))
        })

schema_df = pd.DataFrame(schema_records)

display(schema_df)

,dataset,column,data_type,non_null_count,null_count,unique_values
0,customers,customer_id,object,99441,0,99441
1,customers,customer_unique_id,object,99441,0,96096
2,customers,customer_zip_code_prefix,int64,99441,0,14994
3,customers,customer_city,object,99441,0,4119
4,customers,customer_state,object,99441,0,27
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0,19015
6,geolocation,geolocation_lat,float64,1000163,0,717360
7,geolocation,geolocation_lng,float64,1000163,0,717613
8,geolocation,geolocation_city,object,1000163,0,8011
9,geolocation,geolocation_state,object,1000163,0,27


In [10]:
for dataset_name, df in datasets.items():

    print("\n" + "=" * 100)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 100)

    dataset_schema = schema_df[
        schema_df["dataset"] == dataset_name
    ]

    display(dataset_schema)


DATASET: CUSTOMERS


,dataset,column,data_type,non_null_count,null_count,unique_values
0,customers,customer_id,object,99441,0,99441
1,customers,customer_unique_id,object,99441,0,96096
2,customers,customer_zip_code_prefix,int64,99441,0,14994
3,customers,customer_city,object,99441,0,4119
4,customers,customer_state,object,99441,0,27



DATASET: GEOLOCATION


,dataset,column,data_type,non_null_count,null_count,unique_values
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0,19015
6,geolocation,geolocation_lat,float64,1000163,0,717360
7,geolocation,geolocation_lng,float64,1000163,0,717613
8,geolocation,geolocation_city,object,1000163,0,8011
9,geolocation,geolocation_state,object,1000163,0,27



DATASET: ORDER_ITEMS


...

,dataset,column,data_type,non_null_count,null_count,unique_values
37,products,product_id,object,32951,0,32951
38,products,product_category_name,object,32341,610,73
39,products,product_name_lenght,float64,32341,610,66
40,products,product_description_lenght,float64,32341,610,2960
41,products,product_photos_qty,float64,32341,610,19
42,products,product_weight_g,float64,32949,2,2204
43,products,product_length_cm,float64,32949,2,99
44,products,product_height_cm,float64,32949,2,102
45,products,product_width_cm,float64,32949,2,95



DATASET: SELLERS


,dataset,column,data_type,non_null_count,null_count,unique_values
46,sellers,seller_id,object,3095,0,3095
47,sellers,seller_zip_code_prefix,int64,3095,0,2246
48,sellers,seller_city,object,3095,0,611
49,sellers,seller_state,object,3095,0,23



DATASET: CATEGORY_TRANSLATION


,dataset,column,data_type,non_null_count,null_count,unique_values
50,category_translation,product_category_name,object,71,0,71
51,category_translation,product_category_name_english,object,71,0,71


In [11]:
for dataset_name, df in datasets.items():

    print("\n" + "=" * 100)
    print(f"SAMPLE DATA: {dataset_name.upper()}")
    print("=" * 100)

    display(df.head(5))


SAMPLE DATA: CUSTOMERS


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



SAMPLE DATA: GEOLOCATION


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP



SAMPLE DATA: ORDER_ITEMS


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00



SAMPLE DATA: SELLERS


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



SAMPLE DATA: CATEGORY_TRANSLATION


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [12]:
for dataset_name, df in datasets.items():

    print("\n" + "=" * 100)
    print(f"DATASET INFORMATION: {dataset_name.upper()}")
    print("=" * 100)

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("\nData Types:")
    print(df.dtypes)


DATASET INFORMATION: CUSTOMERS
Shape: 99,441 rows × 5 columns

Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

DATASET INFORMATION: GEOLOCATION
Shape: 1,000,163 rows × 5 columns

Data Types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

DATASET INFORMATION: ORDER_ITEMS
Shape: 112,650 rows × 7 columns

Data Types:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

DATASET INFORMATION: ORDER_PAYMENTS
Shape: 103,886 rows × 5 columns

Data Types:
order_id                 object
payment_sequential   

In [13]:
def calculate_missing_values(
    datasets: Dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """
    Calculate missing values and missing percentages
    for every column in every dataset.
    """
    records = []

    for dataset_name, df in datasets.items():

        total_rows = len(df)

        for column in df.columns:

            missing_count = df[column].isna().sum()
            missing_percentage = (
                (missing_count / total_rows) * 100
                if total_rows > 0
                else 0
            )

            records.append({
                "dataset": dataset_name,
                "column": column,
                "total_rows": total_rows,
                "missing_count": int(missing_count),
                "missing_percentage": round(
                    missing_percentage,
                    2
                )
            })

    return pd.DataFrame(records)


missing_analysis = calculate_missing_values(datasets)

display(
    missing_analysis.sort_values(
        ["missing_percentage", "missing_count"],
        ascending=False
    )
)

,dataset,column,total_rows,missing_count,missing_percentage
25,order_reviews,review_comment_title,99224,87656,88.34
26,order_reviews,review_comment_message,99224,58247,58.70
35,orders,order_delivered_customer_date,99441,2965,2.98
38,products,product_category_name,32951,610,1.85
39,products,product_name_lenght,32951,610,1.85
40,products,product_description_lenght,32951,610,1.85
41,products,product_photos_qty,32951,610,1.85
34,orders,order_delivered_carrier_date,99441,1783,1.79
33,orders,order_approved_at,99441,160,0.16
42,products,product_weight_g,32951,2,0.01


In [14]:
columns_with_missing = (
    missing_analysis[
        missing_analysis["missing_count"] > 0
    ]
    .sort_values(
        ["missing_percentage", "missing_count"],
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    f"Columns containing missing values: "
    f"{len(columns_with_missing)}"
)

display(columns_with_missing)

Columns containing missing values: 13


,dataset,column,total_rows,missing_count,missing_percentage
0,order_reviews,review_comment_title,99224,87656,88.34
1,order_reviews,review_comment_message,99224,58247,58.70
2,orders,order_delivered_customer_date,99441,2965,2.98
3,products,product_category_name,32951,610,1.85
4,products,product_name_lenght,32951,610,1.85
5,products,product_description_lenght,32951,610,1.85
6,products,product_photos_qty,32951,610,1.85
7,orders,order_delivered_carrier_date,99441,1783,1.79
8,orders,order_approved_at,99441,160,0.16
9,products,product_weight_g,32951,2,0.01


In [15]:
dataset_missing_summary = (
    missing_analysis
    .groupby("dataset", as_index=False)
    .agg(
        total_missing_values=("missing_count", "sum"),
        columns_with_missing=("missing_count", lambda x: (x > 0).sum()),
        total_columns=("column", "count")
    )
)

dataset_missing_summary["columns_with_missing_percentage"] = (
    dataset_missing_summary["columns_with_missing"]
    / dataset_missing_summary["total_columns"]
    * 100
).round(2)

dataset_missing_summary = dataset_missing_summary.sort_values(
    "total_missing_values",
    ascending=False
)

display(dataset_missing_summary)

,dataset,total_missing_values,columns_with_missing,total_columns,columns_with_missing_percentage
5,order_reviews,145903,2,7,28.57
6,orders,4908,3,8,37.50
7,products,2448,8,9,88.89
0,category_translation,0,0,2,0.00
1,customers,0,0,5,0.00
2,geolocation,0,0,5,0.00
3,order_items,0,0,7,0.00
4,order_payments,0,0,5,0.00
8,sellers,0,0,4,0.00


In [16]:
def calculate_duplicate_rows(
    datasets: Dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """
    Calculate exact duplicate rows for every dataset.
    """
    records = []

    for dataset_name, df in datasets.items():

        duplicate_count = df.duplicated().sum()

        duplicate_percentage = (
            duplicate_count / len(df) * 100
            if len(df) > 0
            else 0
        )

        records.append({
            "dataset": dataset_name,
            "total_rows": len(df),
            "duplicate_rows": int(duplicate_count),
            "duplicate_percentage": round(
                duplicate_percentage,
                4
            )
        })

    return pd.DataFrame(records)


duplicate_analysis = calculate_duplicate_rows(datasets)

display(
    duplicate_analysis.sort_values(
        "duplicate_rows",
        ascending=False
    )
)

,dataset,total_rows,duplicate_rows,duplicate_percentage
1,geolocation,1000163,261831,26.18
0,customers,99441,0,0.00
2,order_items,112650,0,0.00
3,order_payments,103886,0,0.00
4,order_reviews,99224,0,0.00
5,orders,99441,0,0.00
6,products,32951,0,0.00
7,sellers,3095,0,0.00
8,category_translation,71,0,0.00


In [17]:
for dataset_name, df in datasets.items():

    duplicate_rows = df[df.duplicated(keep=False)]

    print("\n" + "=" * 100)
    print(f"DUPLICATE ANALYSIS: {dataset_name.upper()}")
    print("=" * 100)

    print(
        f"Exact duplicate rows: "
        f"{df.duplicated().sum():,}"
    )

    if not duplicate_rows.empty:
        display(duplicate_rows.head(10))
    else:
        print("No exact duplicate rows found.")


DUPLICATE ANALYSIS: CUSTOMERS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: GEOLOCATION
Exact duplicate rows: 261,831


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
6,1047,-23.55,-46.64,sao paulo,SP
7,1013,-23.55,-46.63,sao paulo,SP
8,1029,-23.54,-46.63,sao paulo,SP
9,1011,-23.55,-46.64,sao paulo,SP
10,1013,-23.55,-46.63,sao paulo,SP
13,1012,-23.55,-46.63,sao paulo,SP
15,1046,-23.55,-46.64,sao paulo,SP



DUPLICATE ANALYSIS: ORDER_ITEMS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: ORDER_PAYMENTS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: ORDER_REVIEWS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: ORDERS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: PRODUCTS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: SELLERS
Exact duplicate rows: 0
No exact duplicate rows found.

DUPLICATE ANALYSIS: CATEGORY_TRANSLATION
Exact duplicate rows: 0
No exact duplicate rows found.


In [18]:
KEY_DEFINITIONS = {
    "customers": {
        "type": "primary",
        "columns": ["customer_id"]
    },
    "orders": {
        "type": "primary",
        "columns": ["order_id"]
    },
    "products": {
        "type": "primary",
        "columns": ["product_id"]
    },
    "sellers": {
        "type": "primary",
        "columns": ["seller_id"]
    },
    "order_items": {
        "type": "composite",
        "columns": ["order_id", "order_item_id"]
    },
    "order_payments": {
        "type": "composite",
        "columns": ["order_id", "payment_sequential"]
    }
}

In [19]:
def validate_keys(
    datasets: Dict[str, pd.DataFrame],
    key_definitions: Dict
) -> pd.DataFrame:
    """
    Validate primary and composite keys by checking:

    - Missing key values
    - Duplicate key combinations
    - Unique key combinations
    """
    results = []

    for dataset_name, key_info in key_definitions.items():

        df = datasets[dataset_name]
        key_columns = key_info["columns"]

        missing_key_values = df[key_columns].isna().any(axis=1).sum()

        duplicate_key_rows = df.duplicated(
            subset=key_columns
        ).sum()

        unique_key_count = len(
            df[key_columns].drop_duplicates()
        )

        is_valid = (
            missing_key_values == 0
            and duplicate_key_rows == 0
        )

        results.append({
            "dataset": dataset_name,
            "key_type": key_info["type"],
            "key_columns": ", ".join(key_columns),
            "total_rows": len(df),
            "unique_key_combinations": unique_key_count,
            "missing_key_values": int(missing_key_values),
            "duplicate_key_combinations": int(
                duplicate_key_rows
            ),
            "key_valid": is_valid
        })

    return pd.DataFrame(results)


key_validation = validate_keys(
    datasets,
    KEY_DEFINITIONS
)

display(key_validation)

,dataset,key_type,key_columns,total_rows,unique_key_combinations,missing_key_values,duplicate_key_combinations,key_valid
0,customers,primary,customer_id,99441,99441,0,0,True
1,orders,primary,order_id,99441,99441,0,0,True
2,products,primary,product_id,32951,32951,0,0,True
3,sellers,primary,seller_id,3095,3095,0,0,True
4,order_items,composite,"order_id, order_item_id",112650,112650,0,0,True
5,order_payments,composite,"order_id, payment_sequential",103886,103886,0,0,True


In [20]:
review_id_analysis = pd.DataFrame({
    "metric": [
        "total_rows",
        "unique_review_ids",
        "duplicate_review_ids",
        "missing_review_ids"
    ],
    "value": [
        len(order_reviews),
        order_reviews["review_id"].nunique(),
        order_reviews["review_id"].duplicated().sum(),
        order_reviews["review_id"].isna().sum()
    ]
})

display(review_id_analysis)

,metric,value
0,total_rows,99224
1,unique_review_ids,98410
2,duplicate_review_ids,814
3,missing_review_ids,0


In [21]:
RELATIONSHIPS = [
    {
        "relationship": "orders -> customers",
        "child_dataset": "orders",
        "child_key": "customer_id",
        "parent_dataset": "customers",
        "parent_key": "customer_id"
    },
    {
        "relationship": "order_items -> orders",
        "child_dataset": "order_items",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "order_items -> products",
        "child_dataset": "order_items",
        "child_key": "product_id",
        "parent_dataset": "products",
        "parent_key": "product_id"
    },
    {
        "relationship": "order_items -> sellers",
        "child_dataset": "order_items",
        "child_key": "seller_id",
        "parent_dataset": "sellers",
        "parent_key": "seller_id"
    },
    {
        "relationship": "order_payments -> orders",
        "child_dataset": "order_payments",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "order_reviews -> orders",
        "child_dataset": "order_reviews",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "products -> category_translation",
        "child_dataset": "products",
        "child_key": "product_category_name",
        "parent_dataset": "category_translation",
        "parent_key": "product_category_name"
    }
]

In [22]:
def validate_referential_integrity(
    datasets: Dict[str, pd.DataFrame],
    relationships: List[Dict]
) -> pd.DataFrame:
    """
    Validate relationships between child and parent tables.
    """
    results = []

    for relationship in relationships:

        child_df = datasets[
            relationship["child_dataset"]
        ]

        parent_df = datasets[
            relationship["parent_dataset"]
        ]

        child_key = relationship["child_key"]
        parent_key = relationship["parent_key"]

        child_values = child_df[
            child_key
        ].dropna()

        parent_values = set(
            parent_df[parent_key].dropna()
        )

        unmatched_mask = ~child_values.isin(parent_values)

        unmatched_records = unmatched_mask.sum()

        unmatched_percentage = (
            unmatched_records
            / len(child_values)
            * 100
            if len(child_values) > 0
            else 0
        )

        results.append({
            "relationship": relationship["relationship"],
            "child_dataset": relationship["child_dataset"],
            "parent_dataset": relationship["parent_dataset"],
            "total_child_records": len(child_values),
            "unmatched_records": int(
                unmatched_records
            ),
            "unmatched_percentage": round(
                unmatched_percentage,
                4
            ),
            "integrity_valid": unmatched_records == 0
        })

    return pd.DataFrame(results)


referential_integrity = (
    validate_referential_integrity(
        datasets,
        RELATIONSHIPS
    )
)

display(referential_integrity)

,relationship,child_dataset,parent_dataset,total_child_records,unmatched_records,unmatched_percentage,integrity_valid
0,orders -> customers,orders,customers,99441,0,0.00,True
1,order_items -> orders,order_items,orders,112650,0,0.00,True
2,order_items -> products,order_items,products,112650,0,0.00,True
3,order_items -> sellers,order_items,sellers,112650,0,0.00,True
4,order_payments -> orders,order_payments,orders,103886,0,0.00,True
5,order_reviews -> orders,order_reviews,orders,99224,0,0.00,True
6,products -> category_translation,products,category_translation,32341,13,0.04,False


In [23]:
for relationship in RELATIONSHIPS:

    child_df = datasets[
        relationship["child_dataset"]
    ]

    parent_df = datasets[
        relationship["parent_dataset"]
    ]

    child_key = relationship["child_key"]
    parent_key = relationship["parent_key"]

    unmatched = child_df[
        ~child_df[child_key].isin(
            parent_df[parent_key]
        )
    ]

    print("\n" + "=" * 100)
    print(
        f"RELATIONSHIP: "
        f"{relationship['relationship']}"
    )
    print("=" * 100)

    print(
        f"Unmatched records: "
        f"{len(unmatched):,}"
    )

    if not unmatched.empty:
        display(unmatched.head(10))


RELATIONSHIP: orders -> customers
Unmatched records: 0

RELATIONSHIP: order_items -> orders
Unmatched records: 0

RELATIONSHIP: order_items -> products
Unmatched records: 0

RELATIONSHIP: order_items -> sellers
Unmatched records: 0

RELATIONSHIP: order_payments -> orders
Unmatched records: 0

RELATIONSHIP: order_reviews -> orders
Unmatched records: 0

RELATIONSHIP: products -> category_translation
Unmatched records: 623


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.00,17.00,14.00,12.00
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.00,16.00,7.00,20.00
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.00,20.00,20.00,20.00
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,"18,500.00",41.00,30.00,41.00
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.00,35.00,7.00,12.00
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,"2,200.00",16.00,2.00,11.00
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.00,16.00,7.00,11.00
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.00,18.00,8.00,14.00
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.00,22.00,14.00,14.00
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,"1,400.00",25.00,25.00,25.00


In [24]:
orders_analysis = orders.copy()
order_items_analysis = order_items.copy()
payments_analysis = order_payments.copy()
reviews_analysis = order_reviews.copy()
customers_analysis = customers.copy()
products_analysis = products.copy()

print("Analysis copies created successfully.")

Analysis copies created successfully.


In [25]:
ORDER_DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in ORDER_DATE_COLUMNS:
    orders_analysis[column] = pd.to_datetime(
        orders_analysis[column],
        errors="coerce"
    )


order_items_analysis["shipping_limit_date"] = pd.to_datetime(
    order_items_analysis["shipping_limit_date"],
    errors="coerce"
)

reviews_analysis["review_creation_date"] = pd.to_datetime(
    reviews_analysis["review_creation_date"],
    errors="coerce"
)

reviews_analysis["review_answer_timestamp"] = pd.to_datetime(
    reviews_analysis["review_answer_timestamp"],
    errors="coerce"
)

print("Date columns converted successfully.")

Date columns converted successfully.


In [26]:
date_conversion_summary = []

for column in ORDER_DATE_COLUMNS:

    date_conversion_summary.append({
        "column": column,
        "min_date": orders_analysis[column].min(),
        "max_date": orders_analysis[column].max(),
        "missing_dates": orders_analysis[column].isna().sum()
    })

date_conversion_df = pd.DataFrame(
    date_conversion_summary
)

display(date_conversion_df)

,column,min_date,max_date,missing_dates
0,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
1,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
2,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
3,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
4,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


In [27]:
total_orders = orders_analysis["order_id"].nunique()

unique_customers = (
    customers_analysis["customer_unique_id"]
    .nunique()
)

total_customer_records = (
    customers_analysis["customer_id"]
    .nunique()
)

total_products = products_analysis[
    "product_id"
].nunique()

total_sellers = sellers[
    "seller_id"
].nunique()

total_product_revenue = (
    order_items_analysis["price"]
    .sum()
)

total_freight_value = (
    order_items_analysis["freight_value"]
    .sum()
)

gross_transaction_value = (
    total_product_revenue
    + total_freight_value
)

average_item_price = (
    order_items_analysis["price"]
    .mean()
)

average_order_payment = (
    payments_analysis
    .groupby("order_id")["payment_value"]
    .sum()
    .mean()
)

business_kpis = pd.DataFrame({
    "metric": [
        "Total Orders",
        "Unique Customers",
        "Customer Records",
        "Total Products",
        "Total Sellers",
        "Product Revenue",
        "Freight Value",
        "Gross Transaction Value",
        "Average Item Price",
        "Average Order Payment"
    ],
    "value": [
        total_orders,
        unique_customers,
        total_customer_records,
        total_products,
        total_sellers,
        total_product_revenue,
        total_freight_value,
        gross_transaction_value,
        average_item_price,
        average_order_payment
    ]
})

display(business_kpis)

,metric,value
0,Total Orders,"99,441.00"
1,Unique Customers,"96,096.00"
2,Customer Records,"99,441.00"
3,Total Products,"32,951.00"
4,Total Sellers,"3,095.00"
5,Product Revenue,"13,591,643.70"
6,Freight Value,"2,251,909.54"
7,Gross Transaction Value,"15,843,553.24"
8,Average Item Price,120.65
9,Average Order Payment,160.99


In [28]:
order_status_summary = (
    orders_analysis["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

order_status_summary["percentage"] = (
    order_status_summary["order_count"]
    / order_status_summary["order_count"].sum()
    * 100
).round(2)

display(order_status_summary)

,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [29]:
dataset_time_range = pd.DataFrame({
    "metric": [
        "First Order",
        "Last Order",
        "Dataset Duration (Days)"
    ],
    "value": [
        orders_analysis[
            "order_purchase_timestamp"
        ].min(),
        orders_analysis[
            "order_purchase_timestamp"
        ].max(),
        (
            orders_analysis[
                "order_purchase_timestamp"
            ].max()
            -
            orders_analysis[
                "order_purchase_timestamp"
            ].min()
        ).days
    ]
})

display(dataset_time_range)

,metric,value
0,First Order,2016-09-04 21:15:19
1,Last Order,2018-10-17 17:30:18
2,Dataset Duration (Days),772


In [30]:
payment_type_summary = (
    payments_analysis
    .groupby("payment_type")
    .agg(
        transaction_count=("order_id", "count"),
        total_payment_value=("payment_value", "sum"),
        average_payment_value=("payment_value", "mean")
    )
    .reset_index()
    .sort_values(
        "total_payment_value",
        ascending=False
    )
)

display(payment_type_summary)

,payment_type,transaction_count,total_payment_value,average_payment_value
1,credit_card,76795,"12,542,084.19",163.32
0,boleto,19784,"2,869,361.27",145.03
4,voucher,5775,"379,436.87",65.70
2,debit_card,1529,"217,989.79",142.57
3,not_defined,3,0.00,0.00


In [31]:
review_score_summary = (
    reviews_analysis
    .groupby("review_score")
    .agg(
        review_count=("review_id", "count")
    )
    .reset_index()
    .sort_values("review_score")
)

review_score_summary["percentage"] = (
    review_score_summary["review_count"]
    / review_score_summary["review_count"].sum()
    * 100
).round(2)

display(review_score_summary)

,review_score,review_count,percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [32]:
customer_state_summary = (
    customers_analysis
    .groupby("customer_state")
    .agg(
        customer_records=("customer_id", "count"),
        unique_customers=("customer_unique_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "unique_customers",
        ascending=False
    )
)

display(customer_state_summary)

,customer_state,customer_records,unique_customers
25,SP,41746,40302
18,RJ,12852,12384
10,MG,11635,11259
22,RS,5466,5277
17,PR,5045,4882
23,SC,3637,3534
4,BA,3380,3277
6,DF,2140,2075
7,ES,2033,1964
8,GO,2020,1952


In [33]:
products_enriched = products_analysis.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

print(
    f"Products before merge: {len(products_analysis):,}"
)

print(
    f"Products after merge : {len(products_enriched):,}"
)

Products before merge: 32,951
Products after merge : 32,951


In [34]:
product_category_summary = (
    order_items_analysis
    .merge(
        products_enriched[
            [
                "product_id",
                "product_category_name",
                "product_category_name_english"
            ]
        ],
        on="product_id",
        how="left"
    )
    .groupby(
        "product_category_name_english",
        dropna=False
    )
    .agg(
        items_sold=("order_item_id", "count"),
        total_product_revenue=("price", "sum"),
        average_item_price=("price", "mean"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
    .sort_values(
        "total_product_revenue",
        ascending=False
    )
)

display(product_category_summary.head(20))

,product_category_name_english,items_sold,total_product_revenue,average_item_price,total_freight
43,health_beauty,9670,"1,258,681.34",130.16,"182,566.73"
70,watches_gifts,5991,"1,205,005.68",201.14,"100,535.93"
7,bed_bath_table,11115,"1,036,988.68",93.30,"204,693.04"
65,sports_leisure,8641,"988,048.97",114.34,"168,607.51"
15,computers_accessories,7827,"911,954.32",116.51,"147,318.08"
39,furniture_decor,8334,"729,762.49",87.56,"172,749.30"
20,cool_stuff,3796,"635,290.85",167.36,"84,039.10"
49,housewares,6964,"632,248.66",90.79,"146,149.11"
5,auto,4235,"592,720.11",139.96,"92,664.21"
42,garden_tools,4347,"485,256.46",111.63,"98,962.75"


In [35]:
seller_performance = (
    order_items_analysis
    .groupby("seller_id")
    .agg(
        items_sold=("order_item_id", "count"),
        orders=("order_id", "nunique"),
        product_revenue=("price", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
    .sort_values(
        "product_revenue",
        ascending=False
    )
)

display(seller_performance.head(20))

,seller_id,items_sold,orders,product_revenue,average_item_price
857,4869f7a5dfa277a7dca6462dcf3b52b2,1156,1132,"229,472.63",198.51
1013,53243585a1d6dc2643021fd1853d8905,410,358,"222,776.05",543.36
881,4a3ca9315b744ce9f8e9374361493884,1987,1806,"200,472.92",100.89
3024,fa1c13f2614d7b5c4749cbc52fecda94,586,585,"194,042.03",331.13
1535,7c67e1448b00f6e969d365cea6b010ab,1364,982,"187,923.89",137.77
1560,7e93a43ef30c4f03f38b393420bc753a,340,336,"176,431.87",518.92
2643,da8622b14eb17ae2831f4ac5b9dab84a,1551,1314,"160,236.57",103.31
1505,7a67c85e85bb2ce8582c35f2203ad736,1171,1160,"141,745.53",121.05
192,1025f0e2d44d7041d6cf58b6550e0bfa,1428,915,"138,968.55",97.32
1824,955fee9216a65b617aa5c0531780ce60,1499,1287,"135,171.70",90.17


In [36]:
delivered_orders = orders_analysis[
    orders_analysis["order_status"] == "delivered"
].copy()

delivered_orders["delivery_days"] = (
    delivered_orders[
        "order_delivered_customer_date"
    ]
    -
    delivered_orders[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

delivered_orders["estimated_delivery_days"] = (
    delivered_orders[
        "order_estimated_delivery_date"
    ]
    -
    delivered_orders[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

delivered_orders["delivery_delay_days"] = (
    delivered_orders[
        "order_delivered_customer_date"
    ]
    -
    delivered_orders[
        "order_estimated_delivery_date"
    ]
).dt.total_seconds() / 86400

delivered_orders["is_late_delivery"] = (
    delivered_orders["delivery_delay_days"] > 0
)

delivery_summary = pd.DataFrame({
    "metric": [
        "Delivered Orders",
        "Average Delivery Days",
        "Median Delivery Days",
        "Late Deliveries",
        "Late Delivery Percentage"
    ],
    "value": [
        len(delivered_orders),
        delivered_orders["delivery_days"].mean(),
        delivered_orders["delivery_days"].median(),
        delivered_orders["is_late_delivery"].sum(),
        delivered_orders["is_late_delivery"].mean() * 100
    ]
})

delivery_summary["value"] = delivery_summary[
    "value"
].apply(
    lambda x: round(x, 2)
    if isinstance(x, (float, np.floating))
    else x
)

display(delivery_summary)

,metric,value
0,Delivered Orders,"96,478.00"
1,Average Delivery Days,12.56
2,Median Delivery Days,10.22
3,Late Deliveries,"7,826.00"
4,Late Delivery Percentage,8.11


In [37]:
delivered_orders = orders_analysis[
    orders_analysis["order_status"] == "delivered"
].copy()

delivered_orders["delivery_days"] = (
    delivered_orders[
        "order_delivered_customer_date"
    ]
    -
    delivered_orders[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

delivered_orders["estimated_delivery_days"] = (
    delivered_orders[
        "order_estimated_delivery_date"
    ]
    -
    delivered_orders[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

delivered_orders["delivery_delay_days"] = (
    delivered_orders[
        "order_delivered_customer_date"
    ]
    -
    delivered_orders[
        "order_estimated_delivery_date"
    ]
).dt.total_seconds() / 86400

delivered_orders["is_late_delivery"] = (
    delivered_orders["delivery_delay_days"] > 0
)

delivery_summary = pd.DataFrame({
    "metric": [
        "Delivered Orders",
        "Average Delivery Days",
        "Median Delivery Days",
        "Late Deliveries",
        "Late Delivery Percentage"
    ],
    "value": [
        len(delivered_orders),
        delivered_orders["delivery_days"].mean(),
        delivered_orders["delivery_days"].median(),
        delivered_orders["is_late_delivery"].sum(),
        delivered_orders["is_late_delivery"].mean() * 100
    ]
})

delivery_summary["value"] = delivery_summary[
    "value"
].apply(
    lambda x: round(x, 2)
    if isinstance(x, (float, np.floating))
    else x
)

display(delivery_summary)

,metric,value
0,Delivered Orders,"96,478.00"
1,Average Delivery Days,12.56
2,Median Delivery Days,10.22
3,Late Deliveries,"7,826.00"
4,Late Delivery Percentage,8.11


In [38]:
customer_order_frequency = (
    customers_analysis[
        ["customer_id", "customer_unique_id"]
    ]
    .merge(
        orders_analysis[
            ["order_id", "customer_id"]
        ],
        on="customer_id",
        how="inner"
    )
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

customer_repeat_summary = pd.DataFrame({
    "metric": [
        "Customers with 1 Order",
        "Customers with More Than 1 Order",
        "Repeat Customer Percentage"
    ],
    "value": [
        (
            customer_order_frequency["total_orders"] == 1
        ).sum(),

        (
            customer_order_frequency["total_orders"] > 1
        ).sum(),

        (
            (
                customer_order_frequency["total_orders"] > 1
            ).mean()
            * 100
        )
    ]
})

display(customer_repeat_summary)

,metric,value
0,Customers with 1 Order,"93,099.00"
1,Customers with More Than 1 Order,"2,997.00"
2,Repeat Customer Percentage,3.12


# Phase 3 Conclusions

## Dataset Structure

The Olist dataset consists of nine relational datasets covering:

- Customers
- Orders
- Order items
- Products
- Sellers
- Payments
- Reviews
- Geographic information
- Product category translations

The `orders` table acts as the central transactional entity.

## Data Quality Baseline

The initial analysis establishes a baseline for:

- Missing values
- Duplicate records
- Key integrity
- Referential integrity

These results will be used by the DataTrust Engine in later phases.

## Business Analytics Potential

The dataset supports analysis of:

1. Product performance
2. Customer behavior
3. Customer retention
4. Geographic demand
5. Payment behavior
6. Seller performance
7. Delivery performance
8. Customer satisfaction

## Next Phase

The next phase will focus on building a structured ETL pipeline and preparing
clean analytical datasets without modifying the original raw data.